In [33]:
import unicodedata
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import svm
from sklearn.metrics import f1_score

In [4]:
_STOPWORDS = stopwords.words("spanish")  # agregar más palabras a esta lista si es necesario
PUNCTUACTION = ";:,.\\-\"'/"
SYMBOLS = "()[]¿?¡!{}~<>|@#"
NUMBERS= "0123456789"
SKIP_SYMBOLS = set(PUNCTUACTION + SYMBOLS)
SKIP_SYMBOLS_AND_SPACES = set(PUNCTUACTION + SYMBOLS + '\t\n\r ')

def normaliza_texto(input_str,
                    punct=False,
                    accents=False,
                    num=False,
                    max_dup=2):
    """
        punct=False (elimina la puntuación, True deja intacta la puntuación)
        accents=False (elimina los acentos, True deja intactos los acentos)
        num= False (elimina los números, True deja intactos los acentos)
        max_dup=2 (número máximo de símbolos duplicados de forma consecutiva, rrrrr => rr)
    """
    
    nfkd_f = unicodedata.normalize('NFKD', input_str)
    n_str = []
    c_prev = ''
    cc_prev = 0
    for c in nfkd_f:
        if not num:
            if c in NUMBERS:
                continue
        if not punct:
            if c in SKIP_SYMBOLS:
                continue
        if not accents and unicodedata.combining(c):
            continue
        if c_prev == c:
            cc_prev += 1
            if cc_prev >= max_dup:
                continue
        else:
            cc_prev = 0
        n_str.append(c)
        c_prev = c
    texto = unicodedata.normalize('NFKD', "".join(n_str))
    texto = re.sub(r'(\s)+', r' ', texto.strip(), flags=re.IGNORECASE)
    return texto

def mi_preprocesamiento(texto):
    #convierte a minúsculas el texto antes de normalizar
    # print("antes: ", texto)
    texto = normaliza_texto(texto.lower())
    # print("después:",texto)
    return texto

def mi_tokenizador(texto):
    # Elimina stopwords: palabras que no se consideran de contenido y que no agregan valor semántico al texto
    #print("antes: ", texto)
    texto = [t for t in texto.split() if t not in _STOPWORDS]
    #print("después:",texto)
    return texto

In [20]:
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)

In [21]:

X_train = dataset_train["text"].to_numpy()
Y_train = dataset_train["klass"].to_numpy()


In [22]:
X_test = dataset_test["text"].to_numpy()
Y_test = dataset_test["klass"].to_numpy()

In [35]:
# TODO: Codificar las clases si no son categorías ordinales.

le = LabelEncoder()

Y_encoded= le.fit_transform(Y_train)
Y_test_encoded = le.transform(Y_test)


In [ ]:
vec_tfidf = TfidfVectorizer(analyzer="word", preprocessor=mi_preprocesamiento, tokenizer=mi_tokenizador,  ngram_range=(1,1))
X_tfidf = vec_tfidf.fit_transform(X_train)
X_test_tfidf = vec_tfidf.transform(X_test)

d:\Apps\envs\RNA\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [36]:
X_t = X_test_tfidf
Y_t = Y_test_encoded

In [37]:
X_tr = X_tfidf
Y_tr = Y_encoded

clf = svm.SVC(kernel="linear", random_state=64)
clf.fit(X_tr, Y_tr)
y_pred = clf.predict(X_t)

In [39]:
score = f1_score(Y_t, y_pred, average="macro")
print(f"Desempeño del modelo en el conjunto de test: {score}")

Desempeño del modelo en el conjunto de test: 0.666815183937791
